In [ ]:
# for use in databricks portal the project root path needs to be reference to find the modules in local folder "src"

import os
import sys

current_dir = os.getcwd()
project_root_path = os.path.abspath(os.path.join(current_dir, "..", "..", "..")) # 3 levels up the current dir
# print(project_root_path)
sys.path.append(project_root_path)

In [ ]:
from src.citibike.citibike_utils import get_trip_duration_mins
from src.utils.datetime_utils import timestamp_to_date_col
from pyspark.sql.functions import create_map, lit

In [ ]:
pipeline_id = dbutils.widgets.get("pipeline_id")
run_id = dbutils.widgets.get("run_id")
task_id = dbutils.widgets.get("task_id")
processed_timestamp = dbutils.widgets.get("processed_timestamp")
catalog = dbutils.widgets.get("catalog")

In [ ]:
df = spark.read.table(f"{catalog}.`01_bronze`.jc_citibike")

In [39]:
df = get_trip_duration_mins(spark, df, "started_at", "ended_at", "trip_duration_mins")

In [40]:
df = timestamp_to_date_col(spark, df, "started_at", "trip_start_date")

In [ ]:
df = df.withColumn(
    "meta_data", 
    create_map(
        lit("pipeline_id"), lit(pipeline_id),
        lit("run_id"), lit(run_id),
        lit("task_id"), lit(task_id),
        lit("processed_timestamp"), lit(processed_timestamp),
    )
)

In [ ]:
df = df.select(
    "ride_id",
    "trip_start_date",
    "started_at",
    "ended_at",
    "start_station_name",
    "end_station_name",
    "trip_duration_mins",
    "meta_data"
)

In [ ]:
(
    df.write
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{catalog}.`02_silver`.jc_citibike")
)

In [44]:
# df.show()